# Prophet Forecast

Fit Prophet on the training observations and predict the exact dates in the shared test window.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"


In [ ]:
stock_data = pd.read_csv(DATA_DIR / "tcs_stock_data_cleaned.csv", parse_dates=["Date"])
prophet_data = stock_data[["Date", "Close"]].rename(columns={"Date": "ds", "Close": "y"})
split_index = int(len(prophet_data) * 0.8)
train = prophet_data.iloc[:split_index].copy()
test = prophet_data.iloc[split_index:].copy()


In [ ]:
model = Prophet()
model.fit(train)
test_forecast = model.predict(test[["ds"]])
results = pd.DataFrame(
    {
        "Date": test["ds"],
        "Actual": test["y"].to_numpy(),
        "Prophet_Prediction": test_forecast["yhat"].to_numpy(),
    }
)


In [ ]:
mae = mean_absolute_error(results["Actual"], results["Prophet_Prediction"])
rmse = np.sqrt(mean_squared_error(results["Actual"], results["Prophet_Prediction"]))
pd.Series({"MAE": mae, "RMSE": rmse})


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(results["Date"], results["Actual"], label="Actual")
ax.plot(results["Date"], results["Prophet_Prediction"], label="Prophet")
ax.set(title="Prophet Forecast vs Actual", xlabel="Date", ylabel="Closing Price (INR)")
ax.legend()
fig.autofmt_xdate()
plt.show()


In [ ]:
results.to_csv(DATA_DIR / "prophet_predictions.csv", index=False)
full_forecast = model.predict(prophet_data[["ds"]])
full_forecast.to_csv(DATA_DIR / "prophet_full_forecast.csv", index=False)
